In [ ]:
from _init import *

import os
os.environ['CUDA_VISIBLE_DEVICES'] = '1'

In [ ]:
import random, torch, json

from typing import List, Dict
from transformers import PreTrainedTokenizerFast, AutoModelForCausalLM, DataCollatorForSeq2Seq, TrainingArguments, Trainer
from torch.utils.data import Dataset
from peft import LoraConfig, get_peft_model

from ranger.utils import common_utils, json_utils, tokenizer_utils

In [ ]:
seed = 42
common_utils.set_seed(seed)

work_dir = f'/raid/ai/home/jsyang/dev_env/git/repos/ranger'
data_dir = f'{work_dir}/data'
sft_dir = f'{data_dir}/sft'
selected_dir = f'{sft_dir}/selected'
train_dir = f'{sft_dir}/selected_train'
out_dir = f'{work_dir}/outputs/sft/v99_test'

In [ ]:
train_file_path = f'{train_dir}/train_merged.jsonl'
all_datas = json_utils.load_jsonl(train_file_path)

TRAIN_SIZE = 27000

random.shuffle(all_datas)

train_datas = all_datas[:TRAIN_SIZE]
valid_datas = all_datas[TRAIN_SIZE:]

print(f'train_datas size : {len(train_datas)}')
print(f'valid_datas size : {len(valid_datas)}\n')

json_utils.write_jsonl(train_datas, f'{train_dir}/train_{TRAIN_SIZE}.jsonl')
json_utils.write_jsonl(valid_datas, f'{train_dir}/valid_{len(all_datas)-TRAIN_SIZE}.jsonl')

In [ ]:
print(json_utils.to_str(train_datas[0]))
print(json_utils.to_str(train_datas[1]))
print(json_utils.to_str(train_datas[2]))

In [ ]:
# sources = [[train_data['source']] for train_data in train_datas]
# targets = [train_data['target'] for train_data in train_datas]

# print(f'sources len : {len(sources)}')
# print(f'targets len : {len(targets)}\n')

# print(json_utils.to_str(sources[:3]))
# print(json_utils.to_str(targets[:3]))

In [ ]:
model_name = 'meta-llama/Llama-3.1-8B-Instruct'
# model_name = '/raid/ai/home/jsyang/dev_env/git/repos/ranger/outputs/sft/v2_250222/checkpoint-17600_merged'
max_length = 4096
ignore_index = -100

In [ ]:
tokenizer = tokenizer_utils.load_tokenizer(model_name, 'right')

In [ ]:
# print(tokenizer.eos_token_id)
# print(tokenizer.eos_token)
# print(tokenizer.decode(128001))
# print(tokenizer.decode(128009))
# print()

# source = train_datas[0]['source']
# input_ids = tokenizer_utils.tokenize_apply_chat_template_and_truncate(
#     datas=[[source]],
#     tokenizer=tokenizer,
#     max_length=max_length,
#     add_generation_prompt=True
# )[0]
# decoded_source_toks = [tokenizer.decode(tok_id) for tok_id in input_ids]

# print(f'source : {source}\n')
# print(f'input_ids : {input_ids}\n')
# print(f'decoded_source_toks : {decoded_source_toks}\n')

# target = train_datas[0]['target'] + tokenizer.eos_token
# target_ids = tokenizer(target, add_special_tokens=False)["input_ids"]
# decoded_target_toks = [tokenizer.decode(tok_id) for tok_id in target_ids]

# print(f'target : {target}\n')
# print(f'target_ids : {target_ids}\n')
# print(f'decoded_target_toks : {decoded_target_toks}\n')

In [ ]:
class SftDataset(Dataset):
    def __init__(self, datas: list, tokenizer: PreTrainedTokenizerFast, max_length: int, ignore_index=-100):
        super().__init__()
        self._tokenizer = tokenizer
        self._max_length = max_length
        self._ignore_index = ignore_index

        self._datas = []
        self._preprocess(datas)
    

    def _preprocess(self, datas: list):
        sources = [[data['source']] for data in datas]
        targets = [data['target'] + self._tokenizer.eos_token for data in datas]

        source_ids_list = tokenizer_utils.tokenize_apply_chat_template_and_truncate(
            datas=sources,
            tokenizer=self._tokenizer,
            max_length=self._max_length
        )

        target_ids_list = self._tokenizer(targets, add_special_tokens=False)['input_ids']

        for source_ids, target_ids in zip(source_ids_list, target_ids_list):
            input_ids = source_ids + target_ids
            attention_mask = [1] * len(input_ids)
            labels = [self._ignore_index] * len(source_ids) + target_ids

            self._datas.append({
                'input_ids': input_ids,
                'attention_mask': attention_mask,
                'labels': labels
            })
        
        if DEBUG.TRAIN:
            print(f'# [TRAIN] SftDataset._preprocess() datas size : {len(datas)} -> preprocessed size : {len(self._datas)}')
    

    def __len__(self):
        return len(self._datas)


    def __getitem__(self, index):
        return self._datas[index]

In [ ]:
train_dataset = SftDataset(train_datas, tokenizer, max_length, ignore_index)
valid_dataset = SftDataset(valid_datas, tokenizer, max_length, ignore_index)

In [ ]:
# RangerTrainer 의 초기화 방식과 일치시켜야 함
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=MODEL_CONFIG['dtype'],
    device_map='auto',
    trust_remote_code=False,
    attn_implementation='flash_attention_2'
)

In [ ]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,
    label_pad_token_id=ignore_index
)

In [ ]:
lora_r = 64
lora_alpha = lora_r * 2
lora_dropout = 0.05
target_modules = ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']
bias = 'none'
task_type = 'CAUSAL_LM'

lora_config = LoraConfig(
    r=lora_r,
    lora_alpha=lora_alpha,
    lora_dropout=lora_dropout,
    target_modules=target_modules,
    bias=bias,
    task_type=task_type
)

In [ ]:
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# [핵심 파라미터]
num_epochs = 10
batch_size = 2
accumulation_steps = 16
learning_rate = 5e-5
weight_decay = 0.01
warmup_ratio = 0.05
max_grad_norm = 1.0

# [평가 및 로깅 전략]
save_strategy = 'steps'
save_steps = 200
eval_strategy = 'steps'
eval_steps = 200
logging_steps = 10

# [추가 필수 파라미터]
lr_scheduler_type = 'cosine'        # 학습률을 곡선 형태로 부드럽게 낮춰주어 후반부 미세 조정에 유리함
load_best_model_at_end = True       # 🌟 학습 종료 시, 가장 Valid Loss가 낮았던 "최고 성능" 모델을 자동으로 불러옴
metric_for_best_model = 'eval_loss' # 어떤 지표로 최고 모델을 꼽을지 결정 (Valid Loss 기준)
save_total_limit = 10               # 하드디스크 용량 폭발 방지 (최근 모델과 최고 모델 합쳐서 N개만 유지)

training_args = TrainingArguments(
    output_dir=out_dir,
    num_train_epochs=num_epochs,
    per_device_train_batch_size=batch_size,
    gradient_accumulation_steps=accumulation_steps,
    learning_rate=learning_rate,
    weight_decay=weight_decay,
    warmup_ratio=warmup_ratio,
    max_grad_norm=max_grad_norm,

    save_strategy=save_strategy,
    save_steps=save_steps,
    eval_strategy=eval_strategy,
    eval_steps=eval_steps,
    logging_steps=logging_steps,

    lr_scheduler_type=lr_scheduler_type,
    load_best_model_at_end=load_best_model_at_end,
    metric_for_best_model=metric_for_best_model,
    save_total_limit=save_total_limit,

    bf16=True,
    do_train=True,
    label_names=['labels'],
    report_to='none'
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
    data_collator=data_collator
)

In [ ]:
trainer.train()

In [ ]:
def evaluate_checkpoint(ckpt_path, val_datas, debug_cnt=-1):
    print(f"\n🔄 모델 로드 중: {ckpt_path}")

    total_count = len(val_datas)
    print(f'total_count : {total_count}')
    
    # 평가/추론 시에는 반드시 'left' 패딩
    tokenizer = tokenizer_utils.load_tokenizer(model_name, 'left')

    model = AutoModelForCausalLM.from_pretrained(
        ckpt_path,
        torch_dtype=MODEL_CONFIG['dtype'],
        device_map='auto'
    )
    model.eval()

    em_cnt = 0
    start_cnt = 0
    diff_cnt = 0

    '''
        위에 선언된 토크나이저로 SftDataset 클래스를 다시 만들어야 함
        현재는 전체 코드 상단에 위치한 전역 토크나이저로 생성한 val_datas(SftDataset) 으로 평가하고 있음
        물론, 토크나이저가 바뀌진 않았을거라... 크게 상관은 없을 것 같긴한데
        일단은 정식 코드에는 모두 정상 반영되어 있음
        그리고, 아래는 정답 비교할 때 대/소문자 구분해서 완전 똑같을 때만 하도록 되어 있는데
        대/소문자만 통일해도 성능 엄청 오름 (50% -> 90%)
        마찬가지로 정식 코드에는 반영되어 있음
    '''

    with torch.no_grad():
        for i, val_data in enumerate(val_datas):
            input_ids = val_data['input_ids']
            labels = val_data['labels']
            # print(f'input_ids : {input_ids}')
            # print(f'labels : {labels}')

            source_length = sum(1 for x in labels if x == ignore_index)
            source_ids = input_ids[:source_length]
            target_ids = [x for x in labels if x != ignore_index]
            # print(f'source_length : {source_length}')
            # print(f'source_ids : {source_ids}')
            # print(f'target_ids : {target_ids}')

            source_tensor = torch.tensor([source_ids]).to(model.device)

            attention_mask_tensor = torch.tensor([[1] * source_length]).to(model.device)

            outputs = model.generate(
                input_ids=source_tensor,
                attention_mask=attention_mask_tensor,
                max_new_tokens=64,
                pad_token_id=tokenizer.eos_token_id,
                do_sample=False # Greedy 디코딩 (일관된 평가를 위해)
            )

            generated_ids = outputs[0][source_length:]
            generated_text = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()
            target_text = tokenizer.decode(target_ids, skip_special_tokens=True).strip()
            # print(f'generated_ids : {generated_ids}')
            # print(f'generated_text : {generated_text}')
            # print(f'target_text : {target_text}')

            if generated_text == target_text:
                em_cnt += 1
            elif generated_text.startswith(target_text):
                start_cnt += 1
            else:
                diff_cnt += 1
                if diff_cnt <= debug_cnt:
                    print(f'generated_text : {generated_text}')
                    print(f'target_text : {target_text}\n')

            
            # if i < 10:
            #     print(f'generated_text : {generated_text}')
            #     print(f'target_text : {target_text}\n')
    
    accuracy = (em_cnt / total_count) * 100
    print(f"\n🎉 {os.path.basename(ckpt_path)} EM 정확도: {accuracy:.2f}% ({em_cnt}/{total_count})")

    accuracy = (start_cnt / total_count) * 100
    print(f"\n🎉 {os.path.basename(ckpt_path)} STARTS_WITH 정확도: {accuracy:.2f}% ({start_cnt}/{total_count})")

    full_cnt = em_cnt + start_cnt
    accuracy = (full_cnt / total_count) * 100
    print(f"\n🎉 {os.path.basename(ckpt_path)} FULL 정확도: {accuracy:.2f}% ({full_cnt}/{total_count})")

    del model
    del tokenizer
    torch.cuda.empty_cache()
    
    return accuracy

In [ ]:
import glob

checkpoint_paths = glob.glob(f'{out_dir}/checkpoint-*')
checkpoint_paths.sort(key=lambda x: int(x.split('-')[-1]))

for checkpoint_path in checkpoint_paths:
    evaluate_checkpoint(checkpoint_path, valid_dataset, 10)

In [ ]:
from peft import PeftModel

base_model_path = 'meta-llama/Llama-3.2-3B-Instruct'
adapter_path = '/raid/ai/home/jsyang/dev_env/git/repos/ranger/outputs/sft/v2_250222/checkpoint-17600'
merged_path = '/raid/ai/home/jsyang/dev_env/git/repos/ranger/outputs/sft/v2_250222/checkpoint-17600_merged'

# RangerTrainer 의 초기화 방식과 일치시켜야 함
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_path,
    torch_dtype=MODEL_CONFIG['dtype'],
    device_map='auto',
    trust_remote_code=False,
    attn_implementation='flash_attention_2'
)

model = PeftModel.from_pretrained(base_model, adapter_path)

model = model.merge_and_unload()

model.save_pretrained(merged_path, safe_serialization=True)

tokenizer = tokenizer_utils.load_tokenizer(model_name, 'right')
tokenizer.save_pretrained(merged_path)

In [ ]:
merged_path = '/raid/ai/home/jsyang/dev_env/git/repos/ranger/outputs/sft/v2_250222/checkpoint-17600_merged'
evaluate_checkpoint(merged_path, valid_dataset, 10)